In [ ]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

con = duckdb.connect("../data/warehouse.duckdb", read_only=True)
print("Connected to DuckDB")

In [ ]:
con.sql("""
    SELECT 
        COUNT(*) AS rows,
        MIN(ts_utc) AS start_date,
        MAX(ts_utc) AS end_date,
        COUNT(DISTINCT DATE_TRUNC('day', ts_utc)) AS unique_days,
        COUNT(DISTINCT zone_code) AS zones
    FROM prices_day_ahead
""").df()

In [ ]:
con.sql("""
    SELECT 
        DATE_TRUNC('day', ts_utc) AS day,
        COUNT(*) AS hourly_rows,
        SUM(CASE WHEN price_eur_mwh IS NULL THEN 1 ELSE 0 END) AS missing_prices,
        SUM(CASE WHEN price_eur_mwh < -500 OR price_eur_mwh > 3000 THEN 1 ELSE 0 END) AS extreme_outliers
    FROM prices_day_ahead
    GROUP BY 1
    HAVING hourly_rows != 24 OR missing_prices > 0 OR extreme_outliers > 0
    ORDER BY 1
""").df()

In [ ]:
df = con.sql("""
    SELECT ts_utc, price_eur_mwh 
    FROM prices_day_ahead 
    ORDER BY ts_utc
""").df()

fig = px.line(
    df, x="ts_utc", y="price_eur_mwh",
    title="DK1 Day-Ahead Spot Price 2024 (hourly)",
    labels={"ts_utc": "Time (UTC)", "price_eur_mwh": "Price (EUR/MWh)"}
)
fig.update_layout(height=400)
fig.show()

In [ ]:
df["hour"] = df["ts_utc"].dt.hour
hourly = df.groupby("hour")["price_eur_mwh"].agg(["mean", "std"]).reset_index()
hourly.columns = ["hour", "avg_price", "std"]

fig = px.line(
    hourly, x="hour", y="avg_price",
    error_y="std",
    title="Average DK1 Price by Hour of Day (2024, with ±1σ band)",
    labels={"hour": "Hour of Day (UTC)", "avg_price": "Avg Price (EUR/MWh)"}
)
fig.update_layout(height=400)
fig.show()